# Export Class GeoPackages for ArcGIS Online

## Purpose

ArcGIS Online (AGOL) can struggle with GeoPackage files — field types like small integers and certain encodings cause errors on upload. This notebook converts each class GeoPackage into **File Geodatabase (FGDB)** format, which AGOL handles reliably.

**What this notebook does for each class (1 through 8):**
1. Loads every layer from the class GeoPackage
2. Converts all number fields to **double** and all text fields to **string** (standard types AGOL expects)
3. Saves all layers into a File Geodatabase (`.gdb` folder)
4. If FGDB export fails, falls back to **Shapefiles** instead
5. Creates a **summary table** showing parcel counts, percentages, and values for Asset 1 and Asset 2

**Output location:** `outputs/agol/` folder

## Setup

Connect to Google Drive (or run locally) and install the required libraries. This is the same setup used in Classes 0–8.

In [ ]:
# === ENVIRONMENT SETUP ===
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
    print("Google Drive connected!")
except Exception:
    BASE_DIR = './'
    print("Running locally.")

DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
AGOL_DIR = os.path.join(OUTPUT_DIR, 'agol')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(AGOL_DIR, exist_ok=True)

print(f"  Base directory: {BASE_DIR}")
print(f"  Data directory: {DATA_DIR}")
print(f"  AGOL export directory: {AGOL_DIR}")

In [ ]:
# Install and load libraries
!pip install geopandas fiona shapely pyproj --quiet

import geopandas as gpd
import pandas as pd
import fiona
import shutil
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

In [ ]:
# Helper functions used by every class cell below

def fix_field_types(gdf):
    """
    Convert all fields to types ArcGIS Online can handle:
      - Any number column becomes double
      - Any text column becomes string
      - Geometry is left alone
    """
    for col in gdf.columns:
        if col == 'geometry':
            continue
        if pd.api.types.is_numeric_dtype(gdf[col]):
            # Convert all numbers to double
            gdf[col] = gdf[col].astype('double')
        else:
            # Convert everything else to string
            gdf[col] = gdf[col].astype(str).replace('None', '').replace('nan', '')
    return gdf


def export_gpkg_to_agol(gpkg_path, class_name):
    """
    Convert all layers in a GeoPackage to File Geodatabase (or Shapefiles as fallback).
    Also builds a summary table for the parcels layer.

    Parameters:
      gpkg_path  - full path to the input .gpkg file
      class_name - short name like 'class_1_exposure' used for the output folder
    """
    if not os.path.exists(gpkg_path):
        print(f"  File not found: {gpkg_path}")
        return None

    # List all layers in the GeoPackage
    layers = fiona.listlayers(gpkg_path)
    print(f"  Layers found: {layers}")

    # Set up output path for the FGDB
    gdb_path = os.path.join(AGOL_DIR, f'{class_name}.gdb')

    # Remove old export if it exists so we start fresh
    if os.path.exists(gdb_path):
        shutil.rmtree(gdb_path)

    # Try FGDB first, fall back to Shapefiles
    use_fgdb = True
    try:
        # Test if FGDB write is supported by writing the first layer
        test_gdf = gpd.read_file(gpkg_path, layer=layers[0])
        test_gdf = fix_field_types(test_gdf)
        test_gdf.to_file(gdb_path, layer=layers[0], driver='FileGDB')
        print(f"  FGDB write supported — exporting to: {gdb_path}")
    except Exception:
        try:
            # Try OpenFileGDB driver instead
            if os.path.exists(gdb_path):
                shutil.rmtree(gdb_path)
            test_gdf.to_file(gdb_path, layer=layers[0], driver='OpenFileGDB')
            print(f"  OpenFileGDB write supported — exporting to: {gdb_path}")
        except Exception:
            # Fall back to Shapefiles
            use_fgdb = False
            shp_dir = os.path.join(AGOL_DIR, class_name)
            if os.path.exists(shp_dir):
                shutil.rmtree(shp_dir)
            os.makedirs(shp_dir, exist_ok=True)
            print(f"  FGDB not available — falling back to Shapefiles in: {shp_dir}")
            # Write the first layer as shapefile
            test_gdf.to_file(os.path.join(shp_dir, f'{layers[0]}.shp'), driver='ESRI Shapefile')

    # Export remaining layers (first layer already written above)
    for layer_name in layers[1:]:
        print(f"    Converting layer: {layer_name}")
        gdf = gpd.read_file(gpkg_path, layer=layer_name)
        gdf = fix_field_types(gdf)

        if use_fgdb:
            gdf.to_file(gdb_path, layer=layer_name, driver='OpenFileGDB', mode='a')
        else:
            shp_dir = os.path.join(AGOL_DIR, class_name)
            gdf.to_file(os.path.join(shp_dir, f'{layer_name}.shp'), driver='ESRI Shapefile')

    print(f"    Converting layer: {layers[0]}")
    print(f"  All layers exported.")

    # Build summary table from the parcels layer
    summary = build_summary(gpkg_path, class_name)
    return summary


def build_summary(gpkg_path, class_name):
    """
    Create a summary table for the parcels layer showing:
      - Total parcels in the study area
      - Total Asset 1 and Asset 2 parcels
      - Percentage of study area for each asset group
      - Total parcel value for each asset group
      - Total structure value for each asset group
    """
    layers = fiona.listlayers(gpkg_path)
    if 'parcels' not in layers:
        print("  No 'parcels' layer found — skipping summary.")
        return None

    parcels = gpd.read_file(gpkg_path, layer='parcels')
    total = len(parcels)

    # Count Asset 1 and Asset 2 parcels
    a1_count = 0
    a2_count = 0
    if 'is_asset_1' in parcels.columns:
        a1_count = int((parcels['is_asset_1'] == 1).sum())
    if 'is_asset_2' in parcels.columns:
        a2_count = int((parcels['is_asset_2'] == 1).sum())

    # Percentages of total study area
    a1_pct = round(100 * a1_count / total, 2) if total > 0 else 0
    a2_pct = round(100 * a2_count / total, 2) if total > 0 else 0

    # Parcel values
    a1_parval = 0
    a2_parval = 0
    if 'parval' in parcels.columns:
        if a1_count > 0:
            a1_parval = float(parcels.loc[parcels['is_asset_1'] == 1, 'parval'].sum())
        if a2_count > 0:
            a2_parval = float(parcels.loc[parcels['is_asset_2'] == 1, 'parval'].sum())

    # Structure values (parval - landval, clamped to 0)
    a1_structval = 0
    a2_structval = 0
    if 'parval' in parcels.columns and 'landval' in parcels.columns:
        parcels['_sv'] = (parcels['parval'] - parcels['landval']).clip(lower=0)
        if a1_count > 0:
            a1_structval = float(parcels.loc[parcels['is_asset_1'] == 1, '_sv'].sum())
        if a2_count > 0:
            a2_structval = float(parcels.loc[parcels['is_asset_2'] == 1, '_sv'].sum())

    # Build the summary as a simple table
    summary = pd.DataFrame({
        'Metric': [
            'Total Parcels in Study Area',
            'Asset 1 Parcels',
            'Asset 2 Parcels',
            'Asset 1 % of Study Area',
            'Asset 2 % of Study Area',
            'Asset 1 Total Parcel Value',
            'Asset 2 Total Parcel Value',
            'Asset 1 Total Structure Value',
            'Asset 2 Total Structure Value'
        ],
        'Value': [
            total,
            a1_count,
            a2_count,
            a1_pct,
            a2_pct,
            a1_parval,
            a2_parval,
            a1_structval,
            a2_structval
        ]
    })

    # Save summary as CSV
    csv_path = os.path.join(AGOL_DIR, f'{class_name}_summary.csv')
    summary.to_csv(csv_path, index=False)
    print(f"  Summary saved to: {csv_path}")

    # Also try saving as DBF for AGOL
    try:
        dbf_path = os.path.join(AGOL_DIR, f'{class_name}_summary.dbf')
        # DBF needs a geometry column so we save as CSV instead — DBF without geometry
        # is not supported by geopandas. CSV is the reliable fallback for tables.
    except Exception:
        pass

    return summary


print("Helper functions ready.")

## Class 1: Exposure

Converts the Class 1 GeoPackage which contains the exposure assessment. This includes the parcels layer with exposure scores (whether each parcel is in the flood zone) plus the flood zone and study area layers.

In [ ]:
# Convert Class 1: Exposure
gpkg_path = os.path.join(DATA_DIR, 'class_1_exposure.gpkg')
print("Class 1: Exposure")
print("=" * 50)
summary_1 = export_gpkg_to_agol(gpkg_path, 'class_1_exposure')
if summary_1 is not None:
    display(summary_1)

## Class 2: Potential Impact

Converts the Class 2 GeoPackage which contains potential impact scores. This adds the `potential_impact_a1` and `potential_impact_a2` columns showing how severe flooding would be for each asset group.

In [ ]:
# Convert Class 2: Potential Impact
gpkg_path = os.path.join(DATA_DIR, 'class_2_impact.gpkg')
print("Class 2: Potential Impact")
print("=" * 50)
summary_2 = export_gpkg_to_agol(gpkg_path, 'class_2_impact')
if summary_2 is not None:
    display(summary_2)

## Class 3: Adaptive Capacity

Converts the Class 3 GeoPackage which contains adaptive capacity scores. This adds `adaptive_capacity_a1` and `adaptive_capacity_a2` columns based on building age relative to flood regulation dates.

In [ ]:
# Convert Class 3: Adaptive Capacity
gpkg_path = os.path.join(DATA_DIR, 'class_3_adaptive_capacity.gpkg')
print("Class 3: Adaptive Capacity")
print("=" * 50)
summary_3 = export_gpkg_to_agol(gpkg_path, 'class_3_adaptive_capacity')
if summary_3 is not None:
    display(summary_3)

## Class 4: Vulnerability

Converts the Class 4 GeoPackage which combines potential impact and adaptive capacity into vulnerability scores. This adds `vulnerability_a1` and `vulnerability_a2` columns.

In [ ]:
# Convert Class 4: Vulnerability
gpkg_path = os.path.join(DATA_DIR, 'class_4_vulnerability.gpkg')
print("Class 4: Vulnerability")
print("=" * 50)
summary_4 = export_gpkg_to_agol(gpkg_path, 'class_4_vulnerability')
if summary_4 is not None:
    display(summary_4)

## Class 5: Probability

Converts the Class 5 GeoPackage which contains flood probability scores based on flood zone categories. This adds `probability_a1` and `probability_a2` columns.

In [ ]:
# Convert Class 5: Probability
gpkg_path = os.path.join(DATA_DIR, 'class_5_probability.gpkg')
print("Class 5: Probability")
print("=" * 50)
summary_5 = export_gpkg_to_agol(gpkg_path, 'class_5_probability')
if summary_5 is not None:
    display(summary_5)

## Class 6: Consequence

Converts the Class 6 GeoPackage which contains consequence scores based on structure values and separate medians for each asset group. This adds `consequence_a1` and `consequence_a2` columns.

In [ ]:
# Convert Class 6: Consequence
gpkg_path = os.path.join(DATA_DIR, 'class_6_consequence.gpkg')
print("Class 6: Consequence")
print("=" * 50)
summary_6 = export_gpkg_to_agol(gpkg_path, 'class_6_consequence')
if summary_6 is not None:
    display(summary_6)

## Class 7: Risk

Converts the Class 7 GeoPackage which combines consequence and probability into risk scores. This adds `risk_a1` and `risk_a2` columns.

In [ ]:
# Convert Class 7: Risk
gpkg_path = os.path.join(DATA_DIR, 'class_7_risk.gpkg')
print("Class 7: Risk")
print("=" * 50)
summary_7 = export_gpkg_to_agol(gpkg_path, 'class_7_risk')
if summary_7 is not None:
    display(summary_7)

## Class 8: Combined Score

Converts the Class 8 GeoPackage which contains the final combined risk scores. This is the last step in the assessment pipeline and adds `combined_a1` and `combined_a2` columns that bring together vulnerability and risk into one overall score per asset group.

In [ ]:
# Convert Class 8: Combined Score
gpkg_path = os.path.join(DATA_DIR, 'class_8_final.gpkg')
print("Class 8: Combined Score")
print("=" * 50)
summary_8 = export_gpkg_to_agol(gpkg_path, 'class_8_final')
if summary_8 is not None:
    display(summary_8)

## Done

All class GeoPackages have been converted. Your exported files are in the `outputs/agol/` folder:

- **FGDB files** (`.gdb` folders) — or Shapefiles if FGDB was not available
- **Summary CSVs** — one per class with parcel counts, percentages, and values for Asset 1 and Asset 2

To upload to ArcGIS Online:
1. Zip each `.gdb` folder (or the shapefile folder)
2. Go to **Content → New Item → Your Device**
3. Upload the zip file
4. AGOL will recognize the layers and field types automatically